# Robust Susceptibility Validation Input Generator

`validation_hist.ipynb` と同じ validation 用の摂動点を、HFSS 実行前の `parametric_input.csv` として作成する notebook です。

## 使い方

1. `x_center = []` に 13 次元の中心パラメータを入力します。
2. `N_VALIDATION = 100`, custom perturbation std (`PERTURBATION_SIGMA`), `RANDOM_SEED = 101` の条件で乱数を生成します。従来の `PERTURBATION_STD_RATIO = 0.01` はコメントとして残しています。`PERTURBATION_ENABLED` で各パラメータを perturbation するか固定するかを選べます。
3. index `0` を中心点、index `1`〜`100` を摂動点とする `101 x 13` の DataFrame を作ります。
4. `_config.toml` の `[hfss]` / `param_groups` から取得した parameter names を columns にします。
5. timestamp 付き run directory を作成し、その中に `parametric_input.csv` を保存します。
6. HFSS から返る frequency sweep の `S11(f)` を simulation ごとに列追加し、frequency は 1 列だけ持つ `s11_frequency_response.csv` / HDF5 dataset として保存します。


In [1]:
from pathlib import Path
import json
import os
import time

import numpy as np
import pandas as pd

import lib_config as config
import lib_backbone
import lib_gp

%load_ext autoreload
%autoreload 2


c:\Users\Suzuki Lab 10\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load parameter metadata from _config.toml.
_config = config._loadConfig(Path("./_config.toml"))
app_config = config.initParams(_config, debug=True)

backbone = lib_backbone.Backbone(config=app_config)
base_dir = app_config.env.dir_base
cfg = app_config.hfss
ROUND_DECIMALS = app_config.runtime.round_decimals

LOWER_BOUNDS = np.asarray(cfg.lower_bounds, dtype=float)
UPPER_BOUNDS = np.asarray(cfg.upper_bounds, dtype=float)
BOUNDS = np.vstack([LOWER_BOUNDS, UPPER_BOUNDS]).T
PARAM_NAMES = list(cfg.param_names)

print("n_params:", cfg.n_params)
print("PARAM_NAMES:", PARAM_NAMES)
print("ROUND_DECIMALS:", ROUND_DECIMALS)



[Io]
  filename_input      : params.csv
  filename_output     : results.csv
  filename_temp       : temp_hfss_export.csv

[Opt]
  kernel_type         : Matern52
  length_scale        : 3.605550
  noise_std           : 0.010000
  noise_var           : 0.000100

[Hfss]
  n_simulation        : 100
  n_repeats           : 1
  n_init              : 20
  n_params            : 13
  lower_bounds        : [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 2, 2, 1]
  upper_bounds        : [10.0, 10.0, 10.0, 10.0, 10.0, 2.0, 2.0, 2.0, 2.0, 2.0, 7, 7, 6]
  param_names         : ['h1', 'h2', 'h3', 'h4', 'h5', 's1', 's2', 's3', 's4', 's5', 'a', 'b', 'k']
  param_units         : ['mm', 'mm', 'mm', 'mm', 'mm', '', '', '', '', '', 'mm', 'mm', '']
  filename_models     : ['Backshort.step', 'Finshape.step']
  param_groups        : {'A': {'param_names': ['h1', 'h2', 'h3', 'h4', 'h5', 's1', 's2', 's3', 's4', 's5'], 'param_units': ['mm', 'mm', 'mm', 'mm', 'mm', '', '', '', '', ''], 'lower_bounds': [1.0, 1.

In [3]:
# ===== User input =====
# cfg.n_params (= 13) と同じ長さの中心パラメータをここに入力してください。
# Example:
# x_center = np.asarray([
#     6.225785,
#     4.065235,
#     9.188137,
#     9.788713,
#     1.000000,
#     1.071733,
#     2.000000,
#     1.456739,
#     1.835538,
#     0.537034,
#     2.000000,
#     3.711260,
#     6.000000,
# ], dtype=float)
x_center = [
    6.4960883213,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0654763355,
    2.0,
    0.0,
    2.0,
    0.0,
    2.0,
    3.6582012393,
    4.397641132,
]

x_center = [
    6.3511177429,
    2.6722608641,
    3.8863096735,
    1.631871185,
    9.1119233092,
    1.038799261,
    1.5701113572,
    0.3666703302,
    0.1712108328,
    1.3509923049,
    4.0599152678,
    3.6613867364,
    2.8819211654,
]

N_VALIDATION = 100
# PERTURBATION_STD_RATIO = 0.01  # legacy ratio-based std; kept for reference.
RANDOM_SEED = 101

# Custom per-parameter perturbation standard deviations.
# PARAM_NAMES is loaded from _config.toml in the previous cell, so this dictionary
# is converted to a diagonal covariance matrix in the same parameter order.
CUSTOM_PERTURBATION_STD = {
    "h1": 0.1,
    "h2": 0.1,
    "h3": 0.1,
    "h4": 0.1,
    "h5": 0.1,
    "s1": 0.01,
    "s2": 0.02,
    "s3": 0.02,
    "s4": 0.02,
    "s5": 0.02,
    "a": 0.1,
    "b": 0.1,
    "k": 0.1,
}

# Perturbation enable flags for each x_center component.
# Set False to keep that parameter fixed at x_center for all perturbation rows.
# Example: h1 を固定して他の変数だけ perturbation する場合は "h1": False にします。
PERTURBATION_ENABLED = {
    "h1": True,
    "h2": False,
    "h3": False,
    "h4": False,
    "h5": False,
    "s1": False,
    "s2": False,
    "s3": False,
    "s4": False,
    "s5": False,
    "a": False,
    "b": False,
    "k": False,
}

CUSTOM_PERTURBATION_STD_ARRAY = np.asarray(
    [
        CUSTOM_PERTURBATION_STD[name] if PERTURBATION_ENABLED[name] else 0.0
        for name in PARAM_NAMES
    ],
    dtype=float,
)
PERTURBATION_SIGMA = np.diag(CUSTOM_PERTURBATION_STD_ARRAY**2)
FIXED_PARAM_NAMES = [name for name in PARAM_NAMES if not PERTURBATION_ENABLED[name]]

In [4]:
def validate_x_center(x, n_params, lower_bounds, upper_bounds):
    """Validate and round the user-provided center vector."""
    x = np.asarray(x, dtype=float).reshape(-1)
    if x.size != n_params:
        raise ValueError(
            f"x_center must have length cfg.n_params={n_params}, but got length {x.size}. "
            "Set x_center before generating parametric_input.csv."
        )

    below = x < lower_bounds
    above = x > upper_bounds
    if np.any(below) or np.any(above):
        bad = [
            (PARAM_NAMES[i], float(x[i]), float(lower_bounds[i]), float(upper_bounds[i]))
            for i in np.where(below | above)[0]
        ]
        raise ValueError(f"x_center has out-of-bounds values: {bad}")

    return np.round(x, ROUND_DECIMALS)


def build_validation_sigma(custom_sigma=None):
    """Build validation covariance.

    By default this notebook uses ``PERTURBATION_SIGMA`` from the user-input
    cell. If ``custom_sigma`` is None, the legacy ratio-based formula can still
    be used by defining ``PERTURBATION_STD_RATIO`` in the user-input cell.
    """
    if custom_sigma is not None:
        sigma = np.asarray(custom_sigma, dtype=float)
    else:
        bounds_width = UPPER_BOUNDS - LOWER_BOUNDS
        perturbation_std = PERTURBATION_STD_RATIO * bounds_width
        sigma = np.diag(perturbation_std ** 2)

    expected_shape = (cfg.n_params, cfg.n_params)
    if sigma.shape != expected_shape:
        raise ValueError(f"Sigma must have shape {expected_shape}, got {sigma.shape}.")
    return sigma


In [5]:
x_center = validate_x_center(x_center, cfg.n_params, LOWER_BOUNDS, UPPER_BOUNDS)
VALIDATION_SIGMA = build_validation_sigma(PERTURBATION_SIGMA)
rng = np.random.default_rng(RANDOM_SEED)

# Generate only perturbation rows here. The center row is prepended as index 0 below.
Z_perturbation = lib_gp.sample_input_perturbations(
    x=x_center,
    Sigma=VALIDATION_SIGMA,
    n_samples=N_VALIDATION,
    bounds=BOUNDS,
    rng=rng,
)
Z_perturbation = np.round(Z_perturbation, ROUND_DECIMALS)

# Enforce fixed-parameter flags exactly after sampling/clipping.
if FIXED_PARAM_NAMES:
    fixed_indices = [PARAM_NAMES.index(name) for name in FIXED_PARAM_NAMES]
    Z_perturbation[:, fixed_indices] = x_center[fixed_indices]


# index 0: center, index 1..N_VALIDATION: perturbations
Z_validation_with_center = np.vstack([x_center, Z_perturbation])
parametric_input_df = pd.DataFrame(Z_validation_with_center, columns=PARAM_NAMES)
parametric_input_df.index.name = "sample_idx"

print("Fixed parameters:", FIXED_PARAM_NAMES if FIXED_PARAM_NAMES else "None")
print("DataFrame shape:", parametric_input_df.shape)
print("Center row index:", parametric_input_df.index[0])
print("Perturbation index range:", f"{parametric_input_df.index[1]}..{parametric_input_df.index[-1]}")
display(parametric_input_df.head())


Fixed parameters: ['h2', 'h3', 'h4', 'h5', 's1', 's2', 's3', 's4', 's5', 'a', 'b', 'k']
DataFrame shape: (101, 13)
Center row index: 0
Perturbation index range: 1..100


,h1,h2,h3,h4,h5,s1,s2,s3,s4,s5,a,b,k
sample_idx,,,,,,,,,,,,,
0,6.351118,2.672261,3.88631,1.631871,9.111923,1.038799,1.570111,0.36667,0.171211,1.350992,4.059915,3.661387,2.881921
1,6.272102,2.672261,3.88631,1.631871,9.111923,1.038799,1.570111,0.36667,0.171211,1.350992,4.059915,3.661387,2.881921
2,6.317909,2.672261,3.88631,1.631871,9.111923,1.038799,1.570111,0.36667,0.171211,1.350992,4.059915,3.661387,2.881921
3,6.399691,2.672261,3.88631,1.631871,9.111923,1.038799,1.570111,0.36667,0.171211,1.350992,4.059915,3.661387,2.881921
4,6.385143,2.672261,3.88631,1.631871,9.111923,1.038799,1.570111,0.36667,0.171211,1.350992,4.059915,3.661387,2.881921


In [6]:
# Create a timestamped run directory using the same Backbone directory convention as the other notebooks.
backbone.mkdir()
run_dir = backbone._get_dir_run()

parametric_input_csv = run_dir / "parametric_input.csv"
parametric_input_df.to_csv(parametric_input_csv, index=False)

print(f"Saved parametric input CSV: {parametric_input_csv}")
print(f"Rows x columns: {parametric_input_df.shape[0]} x {parametric_input_df.shape[1]}")


Created new run directory: T:\RAkizawa\HFSS_C2WR10\src\0610110545
Saved parametric input CSV: T:\RAkizawa\HFSS_C2WR10\src\0610110545\parametric_input.csv
Rows x columns: 101 x 13


In [7]:
# Optional: inspect the saved CSV.
saved_parametric_input_df = pd.read_csv(parametric_input_csv)
display(saved_parametric_input_df.head())


,h1,h2,h3,h4,h5,s1,s2,s3,s4,s5,a,b,k
0,6.351118,2.672261,3.88631,1.631871,9.111923,1.038799,1.570111,0.36667,0.171211,1.350992,4.059915,3.661387,2.881921
1,6.272102,2.672261,3.88631,1.631871,9.111923,1.038799,1.570111,0.36667,0.171211,1.350992,4.059915,3.661387,2.881921
2,6.317909,2.672261,3.88631,1.631871,9.111923,1.038799,1.570111,0.36667,0.171211,1.350992,4.059915,3.661387,2.881921
3,6.399691,2.672261,3.88631,1.631871,9.111923,1.038799,1.570111,0.36667,0.171211,1.350992,4.059915,3.661387,2.881921
4,6.385143,2.672261,3.88631,1.631871,9.111923,1.038799,1.570111,0.36667,0.171211,1.350992,4.059915,3.661387,2.881921


## Run parametric HFSS simulations

The cells below execute a simple parametric search over `parametric_input_df`. No Gaussian-process optimization or susceptibility analysis is performed; every row of the generated DataFrame is evaluated once and saved to `results.h5` in the same `output/repeat_1` format used by `main.ipynb`.


In [8]:
# Initialize the HDF5 storer in the same timestamped run directory.
# If the previous save cell already created run_dir, initStorer reuses it.
backbone.initStorer(mode="w")
run_dir = backbone._get_dir_run()

model_paths, model_paths_str = backbone._get_path_models()
RESULTS_FILE = str(run_dir / Path(_config["io"]["filename_output"]))
TEMP_FILE = str(run_dir / Path(_config["io"]["filename_temp"]))

print("Run directory:", run_dir)
print("HDF5 file:", run_dir / "results.h5")
print("RESULTS_FILE:", RESULTS_FILE)
print("TEMP_FILE:", TEMP_FILE)


HDF5 dataset created at: T:\RAkizawa\HFSS_C2WR10\src\0610110545\results.h5
Run directory: T:\RAkizawa\HFSS_C2WR10\src\0610110545
HDF5 file: T:\RAkizawa\HFSS_C2WR10\src\0610110545\results.h5
RESULTS_FILE: T:\RAkizawa\HFSS_C2WR10\src\0610110545\results.csv
TEMP_FILE: T:\RAkizawa\HFSS_C2WR10\src\0610110545\temp_hfss_export.csv


In [9]:
def round_params(params, decimals=ROUND_DECIMALS):
    return np.round(np.asarray(params, dtype=float).flatten(), decimals)


def round_history_row(row, param_names, decimals=ROUND_DECIMALS):
    rounded = dict(row)
    for name in param_names:
        if name in rounded:
            rounded[name] = float(np.round(rounded[name], decimals))
    for name in ("S11", "S11_min_freq_GHz", "Metric", "gamma"):
        if name in rounded and pd.notna(rounded[name]):
            rounded[name] = float(np.round(rounded[name], decimals))
    return rounded


S11_FREQUENCY_COLUMN = "Freq [GHz]"
S11_CURVE_PREFIX = "S11_sim"
s11_frequency_df = None


def _detect_frequency_s11_columns(df_temp):
    """Detect frequency and S11 columns from the HFSS temp CSV."""
    columns = list(df_temp.columns)
    freq_candidates = [column for column in columns if "freq" in str(column).lower()]
    freq_col = freq_candidates[0] if freq_candidates else columns[0]

    s11_candidates = [
        column
        for column in columns
        if column != freq_col and ("s(" in str(column).lower() or "s11" in str(column).lower())
    ]
    s11_col = s11_candidates[0] if s11_candidates else [column for column in columns if column != freq_col][0]
    return freq_col, s11_col


def getResult(input_params, param_names, temp_hfss_path, result_file_path, sim_id):
    """Read one HFSS frequency sweep and append scalar/curve outputs.

    The HFSS temp CSV is expected to contain a frequency column and an S11 column,
    e.g. ``Freq [GHz]`` and ``dB(S(Port1,Port1)) []``. The frequency grid is
    stored only once in ``s11_frequency_df``; each simulation appends a new S11
    curve column. For compatibility with the existing ``repeat_1`` summary format,
    this function also records the minimum S11 value and its frequency in
    ``results.csv``.
    """
    global s11_frequency_df

    df_temp = pd.read_csv(temp_hfss_path)
    header_flag = not os.path.exists(result_file_path)

    try:
        freq_col, s11_col = _detect_frequency_s11_columns(df_temp)
        freq_values = df_temp[freq_col].to_numpy(dtype=float)
        s11_values = df_temp[s11_col].to_numpy(dtype=float)

        if s11_frequency_df is None:
            s11_frequency_df = pd.DataFrame({S11_FREQUENCY_COLUMN: freq_values})
        else:
            existing_freq = s11_frequency_df[S11_FREQUENCY_COLUMN].to_numpy(dtype=float)
            if len(existing_freq) != len(freq_values) or not np.allclose(existing_freq, freq_values):
                raise ValueError("HFSS frequency grid changed between simulations.")

        curve_column = f"{S11_CURVE_PREFIX}_{int(sim_id):03d}"
        s11_frequency_df[curve_column] = s11_values

        min_idx = int(np.nanargmin(s11_values))
        result_row = dict(zip(param_names, input_params))
        result_row["S11"] = float(s11_values[min_idx])
        result_row["S11_min_freq_GHz"] = float(freq_values[min_idx])

        df_result = pd.DataFrame([result_row])
        df_result.to_csv(result_file_path, mode="a", header=header_flag, index=False)

        try:
            os.remove(temp_hfss_path)
        except OSError:
            pass
        return result_row

    except Exception as e:
        print(f"[Error][getResult] Failed to process result: {e}")
        raise


def target_hfss(_config_temp, sim_id, param_names, params):
    """Evaluate one parameter vector through the existing HFSS watcher workflow."""
    params = round_params(params)
    backbone.call_subroutine(
        _config_temp,
        sim_id,
        param_names,
        params,
        value_fmt=f"{{:.{ROUND_DECIMALS}f}}",
    )
    return getResult(params, param_names, TEMP_FILE, RESULTS_FILE, sim_id)


def costFunctionWrapper(param_names, params):
    params = round_params(params, decimals=ROUND_DECIMALS)
    sim_id = backbone._getSimulationID()
    result_row = target_hfss(_config_temp, sim_id, param_names, params)
    y = float(result_row["S11"])

    row = dict(result_row)
    row["Metric"] = np.nan
    row["gamma"] = np.nan
    row["routine_idx"] = 0
    return y, round_history_row(row, param_names)


In [10]:
# HFSS watcher run-specific config, matching main.ipynb's hand-off format.
_config_temp = {
    "n_simulation": int(len(parametric_input_df)),
    "n_repeats": 1,
    "WATCH_DIR": str(run_dir),
    "INPUT_FILE": str(run_dir / Path(_config["io"]["filename_input"])),
    "MODEL_FILE": model_paths_str,
    "RESULTS_FILE": RESULTS_FILE,
    "TEMP_FILE": TEMP_FILE,
    "DONE_FLAG_FILE": str(run_dir / Path("hfss.done")),
}

done_flag_path = Path(_config_temp["DONE_FLAG_FILE"])
done_flag_path.unlink(missing_ok=True)

with open(base_dir / Path("_config_HFSS.json"), "w") as f:
    json.dump(_config_temp, f, indent=4)

print(f"Temporarily updated '{base_dir / Path('_config_HFSS.json')}' with run-specific WATCH_DIR for HFSS.")
print("Number of debug parametric simulations:", len(parametric_input_df))


Temporarily updated 'T:\RAkizawa\HFSS_C2WR10\src\_config_HFSS.json' with run-specific WATCH_DIR for HFSS.
Number of debug parametric simulations: 101


In [11]:
# Simple parametric search debug run: evaluate only the first 5 rows.
# No GP optimization and no susceptibility analysis are performed in this loop.
parametric_history = []
start = time.perf_counter()

try:
    for sample_idx, (_, row) in enumerate(parametric_input_df.iterrows(), start=1):
        params = row[PARAM_NAMES].to_numpy(dtype=float)
        print(f"[parametric debug] HFSS evaluation {sample_idx}/5 (source rows: {len(parametric_input_df)})")
        _, result_row = costFunctionWrapper(PARAM_NAMES, params)
        parametric_history.append(result_row)

    df_final = pd.DataFrame(parametric_history)
    df_final[PARAM_NAMES] = df_final[PARAM_NAMES].round(ROUND_DECIMALS)
    df_final["S11"] = df_final["S11"].round(ROUND_DECIMALS)

    df_output = backbone._genOutputDataFrame(df_final)
    if "best" in df_output:
        df_output["best"] = df_output["best"].round(ROUND_DECIMALS)

    parametric_results_csv = run_dir / "parametric_results.csv"
    df_output.to_csv(parametric_results_csv, index=False)
    backbone._addNewDatasetToHDF(df_output.select_dtypes(include=[np.number]), "output", "repeat_1")

    if s11_frequency_df is not None:
        s11_frequency_csv = run_dir / "s11_frequency_response.csv"
        s11_frequency_df.to_csv(s11_frequency_csv, index=False)
        backbone._addNewDatasetToHDF(s11_frequency_df.select_dtypes(include=[np.number]), "output", "s11_frequency_response")

    elapsed = time.perf_counter() - start
    best_idx = df_output["S11"].idxmin()
    print("-" * 75)
    print(f"Parametric search finished in {elapsed:.3f} seconds")
    print(f"Best S11: {df_output.loc[best_idx, 'S11']:.10f}")
    print("Best row:")
    display(df_output.loc[[best_idx]])
    print(f"Saved results CSV: {parametric_results_csv}")
    if s11_frequency_df is not None:
        print(f"Saved S11 frequency response CSV: {s11_frequency_csv}")
    print(f"Saved HDF5: {run_dir / 'results.h5'}")

finally:
    done_flag_path = Path(_config_temp["DONE_FLAG_FILE"])
    done_flag_path.touch()

    json_file = base_dir / Path("_config_HFSS.json")
    json_file.unlink(missing_ok=True)
    if backbone.h5f:
        backbone.h5f.close()


[parametric debug] HFSS evaluation 1/5 (source rows: 101)
  > Result received from HFSS.
[parametric debug] HFSS evaluation 2/5 (source rows: 101)
  > Result received from HFSS.
[parametric debug] HFSS evaluation 3/5 (source rows: 101)
  > Result received from HFSS.
[parametric debug] HFSS evaluation 4/5 (source rows: 101)
  > Result received from HFSS.
[parametric debug] HFSS evaluation 5/5 (source rows: 101)
  > Result received from HFSS.
[parametric debug] HFSS evaluation 6/5 (source rows: 101)
  > Result received from HFSS.
[parametric debug] HFSS evaluation 7/5 (source rows: 101)
  > Result received from HFSS.
[parametric debug] HFSS evaluation 8/5 (source rows: 101)
  > Result received from HFSS.
[parametric debug] HFSS evaluation 9/5 (source rows: 101)
  > Result received from HFSS.
[parametric debug] HFSS evaluation 10/5 (source rows: 101)
  > Result received from HFSS.
[parametric debug] HFSS evaluation 11/5 (source rows: 101)
  > Result received from HFSS.
[parametric debug] 

C:\Users\Suzuki Lab 10\AppData\Local\Temp\ipykernel_19116\1714682795.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  s11_frequency_df[curve_column] = s11_values


  > Result received from HFSS.
wrote 101 rows x 19 cols to repeat_1
wrote 51 rows x 102 cols to s11_frequency_response
---------------------------------------------------------------------------
Parametric search finished in 6189.690 seconds
Best S11: -50.3930323653
Best row:


C:\Users\Suzuki Lab 10\AppData\Local\Temp\ipykernel_19116\1714682795.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  s11_frequency_df[curve_column] = s11_values


,h1,h2,h3,h4,h5,s1,s2,s3,s4,s5,a,b,k,S11,S11_min_freq_GHz,Metric,gamma,routine_idx,best
81,6.348879,2.672261,3.88631,1.631871,9.111923,1.038799,1.570111,0.36667,0.171211,1.350992,4.059915,3.661387,2.881921,-50.393032,12.8,NaN,NaN,0,-50.393032


Saved results CSV: T:\RAkizawa\HFSS_C2WR10\src\0610110545\parametric_results.csv
Saved S11 frequency response CSV: T:\RAkizawa\HFSS_C2WR10\src\0610110545\s11_frequency_response.csv
Saved HDF5: T:\RAkizawa\HFSS_C2WR10\src\0610110545\results.h5
